# Supervised Contrastive Learning

## Hyperparams and settings

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from settings import Hyperparams, Settings

SETTINGS_PATH = "../configs/settings.yaml"
HYPERPARAMS_PATH = "../configs/hyperparams.yaml"

SETTINGS = Settings(yaml_path=SETTINGS_PATH)
HYPERPARAMS = Hyperparams(yaml_path=HYPERPARAMS_PATH)

In [ ]:
import torch

device = torch.device(SETTINGS.device)

## Tokenizer

In [ ]:
from transformers import XLMRobertaTokenizer

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

new_tokens = ["<EMAIL>", "<TIME>", "<DATE>", "<URL>", "<USERNAME>"]
tokenizer.add_tokens(new_tokens)

## Data

In [ ]:
import pandas as pd

dataframe = pd.read_csv(SETTINGS.train_dataset_path)

### Loader

In [ ]:
from torch.utils.data import DataLoader

from cleaner import Cleaner
from collate_fn import collate_function
from dataset import VacanciesDataset

cleaner = Cleaner()

train_dataset = VacanciesDataset(dataframe, cleaner.process)
train_loader = DataLoader(
    train_dataset,
    batch_size=HYPERPARAMS.batch_size,
    collate_fn=lambda data: collate_function(data, tokenizer=tokenizer),
    shuffle=True,
)

## Model

In [ ]:
from model import XLMRoBERTaSupCon

model = XLMRoBERTaSupCon(proj_dim=HYPERPARAMS.projection_dimension).to(device)
model.encoder.resize_token_embeddings(len(tokenizer))

### Memory optimization

In [ ]:
model.encoder.config.use_cache = False
model.encoder.gradient_checkpointing_enable()

## Loss and optimizer

In [ ]:
from pytorch_metric_learning.losses import SupConLoss

criterion = SupConLoss(temperature=HYPERPARAMS.temperature)

In [ ]:
from torch import optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

warmup_steps = HYPERPARAMS.warmup_steps
optimizer = optim.AdamW(
    model.parameters(),
    lr=HYPERPARAMS.learning_rate,
    weight_decay=HYPERPARAMS.weight_decay,
)

warmup_scheduler = LinearLR(optimizer, start_factor=HYPERPARAMS.lr_start_factor, total_iters=warmup_steps)
main_scheduler = CosineAnnealingLR(optimizer, T_max=(HYPERPARAMS.epochs - warmup_steps), eta_min=HYPERPARAMS.eta_min)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, main_scheduler],
    milestones=[warmup_steps],
)

## Memory warmup

In [ ]:
ROBERTA_MAX_SEQUENCE_LENGTH = 512

model.train()

warmup_input = {
    "input_ids": torch.randint(
        0, 1000, (HYPERPARAMS.batch_size, ROBERTA_MAX_SEQUENCE_LENGTH), device=device,
    ),
    "attention_mask": torch.ones(
        (HYPERPARAMS.batch_size, ROBERTA_MAX_SEQUENCE_LENGTH), device=device,
    ),
}

optimizer.zero_grad(set_to_none=True)

with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    output = model(**warmup_input)
    labels = torch.randint(0, 1, (HYPERPARAMS.batch_size,), device=device)

    loss = criterion(output["projection"], labels)
    loss.backward()
    optimizer.step()

optimizer.zero_grad(set_to_none=True)

## Train

In [ ]:
import mlflow

mlflow.set_tracking_uri(SETTINGS.tracking_uri)
mlflow.set_experiment(SETTINGS.experiment_name)

if mlflow.active_run():
    mlflow.end_run()

run = mlflow.start_run(run_name=SETTINGS.run_name)

mlflow.log_params(vars(HYPERPARAMS))

In [ ]:
from tqdm.auto import tqdm

from metrics import lalign, lunif
from training import train_step

STAFF_TYPE_MAP = {
    "staff": 0,
    "outstaff": 1,
    "bench": 2,
}

AUGMENTED_CLASS = STAFF_TYPE_MAP["class-2"]

MAX_CLASS_SIZE = dataframe["staff_type"].value_counts().max()
AUGMENTED_CLASS_SIZE = dataframe["staff_type"].value_counts()["class-2"]

K = round(MAX_CLASS_SIZE / AUGMENTED_CLASS_SIZE)

total_epochs = HYPERPARAMS.epochs
global_step = 0

for epoch in tqdm(range(total_epochs)):
    running_loss = 0.0
    epoch_align = 0.0
    epoch_unif = 0.0

    for batch_X, batch_y in train_loader:
        ### Augmentation
        input_ids = batch_X["input_ids"]
        attention_mask = batch_X["attention_mask"]

        augmented_mask = batch_y == AUGMENTED_CLASS
        not_augmented_mask = batch_y != AUGMENTED_CLASS

        augmented_input_ids = input_ids[augmented_mask].repeat(K, 1)
        augmented_attention_mask = attention_mask[augmented_mask].repeat(K, 1)
        augmented_labels = batch_y[augmented_mask].repeat(K)

        not_augmented_input_ids = input_ids[not_augmented_mask]
        not_augmented_attention_mask = attention_mask[not_augmented_mask]
        not_augmented_labels = batch_y[not_augmented_mask]

        input_ids = torch.concat([augmented_input_ids, not_augmented_input_ids])
        attention_mask = torch.concat(
            [augmented_attention_mask, not_augmented_attention_mask]
        )
        labels = torch.concat([augmented_labels, not_augmented_labels])

        indices = torch.randperm(labels.size(0))
        input_ids = input_ids[indices]
        attention_mask = attention_mask[indices]
        labels = labels[indices]
        ###

        output = train_step(
            model,
            optimizer,
            criterion,
            {"input_ids": input_ids, "attention_mask": attention_mask},
            labels,
        )

        running_loss += output["loss"]

        alignment = lalign(output["norm_embeddings"], labels.to(device))
        uniformity = lunif(output["norm_embeddings"])

        epoch_align += alignment
        epoch_unif += uniformity

        mlflow.log_metrics(
            {
                "loss": output["loss"],
                "alignment": alignment,
                "uniformity": uniformity,
                "gradient norm": output["grad_norm"],
            },
            step=global_step,
        )

        global_step += 1

    current_lr = optimizer.param_groups[0]["lr"]
    mlflow.log_metric("learning rate", current_lr, epoch)
    scheduler.step()

    if (epoch + 1) % 1 == 0:
        avg_loss = running_loss / len(train_loader)
        avg_align = epoch_align / len(train_loader)
        avg_unif = epoch_unif / len(train_loader)

        print(f"Epoch [{epoch + 1}/{total_epochs}] Loss: {avg_loss:.4f} | Align: {avg_align:.4f} | Uniform: {avg_unif:.4f}")

In [ ]:
mlflow.end_run()

## Save

In [ ]:
from pathlib import Path

tokenizer.save_pretrained(SETTINGS.model_directory)
torch.save(model.encoder.state_dict(), Path(SETTINGS.model_directory) / "state_dict.pth")

## Clear memory

In [ ]:
import gc

del model

gc.collect()
torch.cuda.empty_cache()